# 第 15 章 切断とトービット: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/econometrics-programming/15-truncated-tobit.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
seed = 0             # 乱数の種
n = 1000             # 標本の大きさ（潜在変数 y* = 1 + 2x + u、u の標準偏差は 2）

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
sys.path[:] = [p for p in sys.path if p != str(_root)]
sys.path.insert(0, str(_root))
_lib = sys.modules.get("lib")
if _lib is not None and Path(getattr(_lib, "__file__", "") or "").resolve() != (_root / "lib" / "__init__.py").resolve():
    for _name in list(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## 打ち切りと切断で、OLS は縮む（真の傾きは 2）

In [ ]:
import numpy as np

from lib.chapter03.multiple import fit_ols
from lib.chapter15.limited import (
    censor,
    draw_latent_sample,
    fit_tobit,
    fit_truncreg,
    truncate,
)

x, latent = draw_latent_sample(n, np.random.default_rng(seed))
observed = censor(latent)
print(f"打ち切られた割合 {(observed == 0).mean():.2f}")
print(f"潜在変数への OLS（本当は見えない） "
      f"{fit_ols(latent, x).coefficients.iloc[1]:.3f}")
print(f"打ち切りデータへの OLS {fit_ols(observed, x).coefficients.iloc[1]:.3f}")
tobit = fit_tobit(observed, x)
print(f"トービット {tobit.coefficients.iloc[1]:.3f}")

x_cut, y_cut = truncate(x, latent)
print(f"切断（残る観測値 {len(y_cut)}）への OLS "
      f"{fit_ols(y_cut, x_cut).coefficients.iloc[1]:.3f}")
print(f"切断回帰 {fit_truncreg(y_cut, x_cut).coefficients.iloc[1]:.3f}")

## 図: 打ち切られたデータと直線

In [ ]:
from lib.chapter15.plot import plot_censored_data

plot_censored_data(x, observed)

## 試してみよう

- 打ち切りデータへの OLS は 1.315 に縮み、トービットは 2.113 で、真の値（2）
  の近くです。`n` を 200 にしても、トービットは 2.048、OLS は 1.135 です
- 切断（観測値が消える）の OLS は 1.356 に縮み、切断回帰は 2.153 です。
  打ち切りと切断では、使う道具が違います